In [1]:
import os, time
import torch
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

# ── GPU selection ──────────────────────────────────────────────────────────────
# Change to your GPU index or MIG partition string
DEVICE_ID = "cuda:5"
device = torch.device(DEVICE_ID if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    idx = int(DEVICE_ID.split(":")[-1]) if ":" in DEVICE_ID else 0
    print("GPU:", torch.cuda.get_device_name(idx))

Image.MAX_IMAGE_PIXELS = None  # allow large pathology images


Using device: cuda:5
GPU: NVIDIA H200 MIG 3g.71gb


## Transform
'resizing all input images to 224×224×3'*
No augmentation here — features saved once, deterministically.

In [2]:
#: resize directly to 224×224 (no crop at extraction time)
# Augmentation (random crop / flip) is applied during model training, not here.
transform = transforms.Compose([
    transforms.Resize((224, 224)),          # paper-exact: 224×224×3
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],         # ImageNet stats (ResNet-50 pretrained)
        std=[0.229, 0.224, 0.225]
    )
])
print("Transform defined (224×224, no augmentation — deterministic extraction)")


Transform defined (224×224, no augmentation — deterministic extraction)


## Dataset
LC25000 — 25,000 images across 5 classes, 5000 each.

In [3]:
DATA_DIR = "lung_colon_image_set"   # ← change to your actual path if different

dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
print(f"Total images : {len(dataset)}")
print(f"Classes found: {dataset.classes}")
print(f"Class→index  : {dataset.class_to_idx}")


Total images : 25001
Classes found: ['Test Set', 'Train and Validation Set']
Class→index  : {'Test Set': 0, 'Train and Validation Set': 1}


## ResNet-50 backbone — stop at layer3
Paper §4.1.1: *'after three stages, generated 14×14×1024 feature maps'*

In [4]:
loader = DataLoader(
    dataset,
    batch_size=128,        # large batch — GPU-bound extraction
    shuffle=False,         # keep original order for index alignment with GNN
    num_workers=8,
    pin_memory=True
)

# ResNet-50 truncated after layer3 → output [B, 1024, 14, 14]
_resnet = models.resnet50(weights="DEFAULT")
cnn_backbone = torch.nn.Sequential(
    _resnet.conv1,
    _resnet.bn1,
    _resnet.relu,
    _resnet.maxpool,
    _resnet.layer1,   # stage 1
    _resnet.layer2,   # stage 2
    _resnet.layer3    # stage 3 → [B, 1024, 14, 14]
    # layer4 NOT included — paper stops here
).to(device).eval()

print("CNN backbone ready  (ResNet-50, stops at layer3)")
print(f"Expected output shape per batch: [B, 1024, 14, 14]")


CNN backbone ready  (ResNet-50, stops at layer3)
Expected output shape per batch: [B, 1024, 14, 14]


## Feature extraction loop
Reshape `[B, 1024, 14, 14]` → `[B, 196, 1024]` (paper Eq.4, §5.2.2 shape for AFF input)

In [5]:
cnn_features_list = []

t0 = time.time()
with torch.no_grad():
    for batch_idx, (images, _) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        feats  = cnn_backbone(images)           # [B, 1024, 14, 14]
        # Reshape to [B, 196, 1024] — matches GNN output shape for AFF fusion
        feats  = feats.flatten(2).permute(0, 2, 1).contiguous()  # [B, 196, 1024]
        cnn_features_list.append(feats.cpu())
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx:>4}/{len(loader)}  shape={feats.shape}")

torch.cuda.synchronize()
elapsed = time.time() - t0
print(f"\nExtraction complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")


  Batch    0/196  shape=torch.Size([128, 196, 1024])
  Batch   10/196  shape=torch.Size([128, 196, 1024])
  Batch   20/196  shape=torch.Size([128, 196, 1024])
  Batch   30/196  shape=torch.Size([128, 196, 1024])
  Batch   40/196  shape=torch.Size([128, 196, 1024])
  Batch   50/196  shape=torch.Size([128, 196, 1024])
  Batch   60/196  shape=torch.Size([128, 196, 1024])
  Batch   70/196  shape=torch.Size([128, 196, 1024])
  Batch   80/196  shape=torch.Size([128, 196, 1024])
  Batch   90/196  shape=torch.Size([128, 196, 1024])
  Batch  100/196  shape=torch.Size([128, 196, 1024])
  Batch  110/196  shape=torch.Size([128, 196, 1024])
  Batch  120/196  shape=torch.Size([128, 196, 1024])
  Batch  130/196  shape=torch.Size([128, 196, 1024])
  Batch  140/196  shape=torch.Size([128, 196, 1024])
  Batch  150/196  shape=torch.Size([128, 196, 1024])
  Batch  160/196  shape=torch.Size([128, 196, 1024])
  Batch  170/196  shape=torch.Size([128, 196, 1024])
  Batch  180/196  shape=torch.Size([128, 196, 

In [6]:
cnn_features = torch.cat(cnn_features_list, dim=0)
print(f"Final CNN feature tensor shape : {cnn_features.shape}")
# Expected: [25000 or 25001, 196, 1024]

assert cnn_features.shape[1] == 196,  "Expected 196 spatial nodes (14×14)"
assert cnn_features.shape[2] == 1024, "Expected 1024 feature channels"
print("Shape assertion passed.")


Final CNN feature tensor shape : torch.Size([25001, 196, 1024])
Shape assertion passed.


## Save
> **Note:** Labels are NOT saved here. The GNN notebook (`lc25000_labels.pt`) produces the authoritative 5-class labels from the correct class folder names.

In [7]:
torch.save(cnn_features, "lc25000_cnn_features_v2.pt")
print(f"Saved → lc25000_cnn_features_v2.pt  {cnn_features.shape}")
print("Done. Do NOT use the ImageFolder labels from this notebook.")
print("Use lc25000_labels.pt generated by the GNN extraction notebook.")


Saved → lc25000_cnn_features_v2.pt  torch.Size([25001, 196, 1024])
Done. Do NOT use the ImageFolder labels from this notebook.
Use lc25000_labels.pt generated by the GNN extraction notebook.
